In [1]:
from pathlib import Path
import pandas as pd

# Locate project folder
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == "ml":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "ml" / "data" / "raw"

# Load inventory and product datasets
products = pd.read_csv(DATA_DIR / "products.csv")
batches = pd.read_csv(DATA_DIR / "inventory_batches.csv")

print("Products shape:", products.shape)
print("Inventory batches shape:", batches.shape)

print("\nProducts columns:")
print(products.columns.tolist())

print("\nInventory batches columns:")
print(batches.columns.tolist())

display(products.head())
display(batches.head())

Products shape: (10, 6)
Inventory batches shape: (2000, 16)

Products columns:
['product_id', 'product_name', 'category', 'shelf_life_days', 'selling_price', 'unit_cost']

Inventory batches columns:
['batch_id', 'product_id', 'category', 'received_at', 'snapshot_date', 'expiry_date', 'initial_quantity', 'quantity_remaining', 'days_to_expiry', 'avg_daily_sales_7d', 'expected_demand_before_expiry', 'unit_price', 'unit_cost', 'discount_pct', 'expiry_risk_label', 'waste_units_actual_simulated']


,product_id,product_name,category,shelf_life_days,selling_price,unit_cost
0,1001,Milk 500ml,Dairy,5,32.0,24.0
1,1002,Yogurt Cup,Dairy,10,25.0,17.0
2,1003,Bread Loaf,Bakery,4,40.0,26.0
3,1004,Paneer 200g,Dairy,7,85.0,62.0
4,1005,Spinach Pack,Produce,3,30.0,18.0


,batch_id,product_id,category,received_at,snapshot_date,expiry_date,initial_quantity,quantity_remaining,days_to_expiry,avg_daily_sales_7d,expected_demand_before_expiry,unit_price,unit_cost,discount_pct,expiry_risk_label,waste_units_actual_simulated
0,5001,1006,Produce,2026-08-14,2026-08-16,2026-08-21,140,72,5,23.38,117,45.0,28.0,5,1,64
1,5002,1006,Produce,2026-07-17,2026-07-23,2026-07-24,140,123,1,36.86,37,45.0,28.0,10,1,31
2,5003,1001,Dairy,2026-08-18,2026-08-20,2026-08-23,22,3,3,12.67,38,32.0,24.0,10,1,3
3,5004,1003,Bakery,2026-08-21,2026-08-26,2026-08-25,141,141,-1,14.52,0,40.0,26.0,10,1,114
4,5005,1007,Produce,2026-07-16,2026-07-19,2026-07-30,139,52,11,13.09,144,120.0,78.0,20,1,34


In [2]:
# Work on a copy of the inventory batches
features_df = batches.copy()

# Convert date columns to datetime
date_columns = ["received_at", "snapshot_date", "expiry_date"]

for col in date_columns:
    features_df[col] = pd.to_datetime(
        features_df[col], errors="coerce"
    )

# Create date-based features
features_df["days_since_received"] = (
    features_df["snapshot_date"] - features_df["received_at"]
).dt.days

features_df["snapshot_month"] = features_df["snapshot_date"].dt.month
features_df["snapshot_day_of_week"] = (
    features_df["snapshot_date"].dt.dayofweek
)

# Calculate remaining-stock ratio safely
features_df["remaining_stock_ratio"] = (
    features_df["quantity_remaining"]
    / features_df["initial_quantity"].replace(0, pd.NA)
)

# Create demand coverage feature
features_df["demand_coverage_ratio"] = (
    features_df["expected_demand_before_expiry"]
    / features_df["quantity_remaining"].replace(0, pd.NA)
)

print("Original rows:", len(batches))
print("Feature dataset shape:", features_df.shape)
print("\nNew features:")
display(features_df[
    [
        "days_since_received",
        "snapshot_month",
        "snapshot_day_of_week",
        "remaining_stock_ratio",
        "demand_coverage_ratio",
    ]
].head())

print("\nMissing values in new features:")
display(features_df[
    [
        "days_since_received",
        "snapshot_month",
        "snapshot_day_of_week",
        "remaining_stock_ratio",
        "demand_coverage_ratio",
    ]
].isna().sum())

Original rows: 2000
Feature dataset shape: (2000, 21)

New features:


C:\Users\Dell\AppData\Local\Temp\ipykernel_38608\2183434420.py:19: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  features_df["snapshot_date"].dt.dayofweek


,days_since_received,snapshot_month,snapshot_day_of_week,remaining_stock_ratio,demand_coverage_ratio
0,2,8,6,0.514286,1.625
1,6,7,3,0.878571,0.300813
2,2,8,3,0.136364,12.666667
3,5,8,2,1.000000,0.0
4,3,7,6,0.374101,2.769231



Missing values in new features:


days_since_received        0
snapshot_month             0
snapshot_day_of_week       0
remaining_stock_ratio      0
demand_coverage_ratio    457
dtype: int64